# 🔥 Fenix Core — دفتر تدريب عقل Fenix الخاص

**الخطوات:**
1. من شريط الأعلى: **Runtime (بيئة التنفيذ) ← Change runtime type (تغيير نوع البيئة) ← T4 GPU ← Save**
2. اضغط زر ▶ على كل خلية بالترتيب (انتظر كل خلية تخلص)
3. آخر خلية يعطيك **ملف fenix-core-lora.zip** — حمّله وأرسله لي

⏱ الوقت الكلي: 30–60 دقيقة تقريباً

## 1️⃣ التحقق من وجود الـ GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "❌ اختر T4 GPU من: Runtime ← Change runtime type ثم أعد تشغيل هذه الخلية"
print("✅ GPU جاهز:", torch.cuda.get_device_name(0))

## 2️⃣ جلب بيانات التدريب من GitHub — اضغط ▶ فقط، بدون رفع أي ملف

In [ ]:
%cd /content
!rm -rf /content/fenix-core
!git clone -q https://github.com/hakarikenji/Fenix-ai /content/fenix-core
import os
if not os.path.exists('/content/fenix-core/training/train_lora.py'):
    print('⚠️ git clone فشل — أجرب الطريقة البديلة (تحميل مباشر)...')
    !curl -sL https://github.com/hakarikenji/Fenix-ai/archive/refs/heads/main.tar.gz -o /content/fc.tgz
    !cd /content && tar xzf fc.tgz && mv Fenix-ai-main fenix-core && rm -f fc.tgz
assert os.path.exists('/content/fenix-core/training/train_lora.py'), '❌ فشل سحب المستودع بطريقتين — تحقق من اتصال الإنترنت وأعد تشغيل هذه الخلية'
!mkdir -p /content/fenix-core/data
!cp /content/fenix-core/training/data/*.jsonl /content/fenix-core/data/
!ls /content/fenix-core/data/
print('✅ المستودع والبيانات جاهزة — تم السحب من GitHub')


## 3️⃣ تثبيت مكتبات التدريب (~دقيقتين)

In [ ]:
%pip install -q -U peft trl bitsandbytes datasets accelerate sentencepiece
%pip install -q 'transformers==4.51.3'
import transformers, peft, datasets
print('✅ المكتبات جاهزة — transformers', transformers.__version__)

## 4️⃣ التدريب (الخلية الطويلة ~30-60 دقيقة)

سيرفع النموذج Qwen3-4B ويضيف عليه "طبقات تعلم" صغيرة (LoRA) — دون المساس بالنموذج الأصلي.
ستشاهد `train_loss` ينزل تدريجياً و `eval_loss` يُقاس كل جولة.

In [ ]:
%cd /content/fenix-core
import os
assert os.path.exists('training/train_lora.py'), '❌ السكربت غير موجود — أعد تشغيل خلية 2 (السحب من GitHub) أولاً'
!python training/train_lora.py --data-dir training/data --out training/runs


## 5️⃣ فحص سريع — جرّب عقلك الجديد مباشرة!

In [ ]:
%cd /content/fenix-core
import json
from pathlib import Path
runs = sorted(Path('training/runs').glob('*/run.json'))
if not runs:
    print('⏭️ أُتخطت هذه الخلية — التدريب لم يكتمل بعد. الرسالة الحمراء الحقيقية موجودة في خلية 4 فوق')
else:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    import torch
    run = runs[-1].parent
    best = json.load(open(run / 'run.json'))['best_checkpoint']
    print('best checkpoint:', best)
    tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-4B-Instruct-2507')
    m = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-4B-Instruct-2507', device_map='auto',
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16))
    m = PeftModel.from_pretrained(m, best)
    def ask(q):
        t = tok.apply_chat_template([{'role':'system','content':'You are Fenix, an AI assistant built by Hakari.'},
                                     {'role':'user','content':q}], tokenize=False, add_generation_prompt=True)
        ids = tok(t, return_tensors='pt').to(m.device)
        out = m.generate(**ids, max_new_tokens=200, do_sample=True, temperature=0.6)
        return tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    print('🧪 Q1:', ask('Who are you and who built you?'))
    print()
    print('🧪 Q2:', ask('من أنت بالضبط ومن بنى you؟'))


## 6️⃣ الحفظ — حمّل ملف النموذج المدرّب

In [ ]:
import os
if not os.path.exists('/content/fenix-core/training/runs') or not os.listdir('/content/fenix-core/training/runs'):
    print('⏭️ أُتخطت هذه الخلية — لا يوجد نموذج مدرَّب بعد. أكمل التدريب أولاً ثم شغّل هذه الخلية')
else:
    !cd /content/fenix-core && zip -rq fenix-core-lora.zip training/runs
    !ls -lh /content/fenix-core/fenix-core-lora.zip
    from google.colab import files
    files.download('/content/fenix-core/fenix-core-lora.zip')
    print('📥 حمّل الملف وأرسله لي — سنوصل العقل بالتطبيق')
